# Agent training

In [ ]:
from stable_baselines3 import PPO, A2C, DQN
from stable_baselines3.common.env_util import make_vec_env
from circle_environment import CircleEnv


# Instantiate the env
# vec_env = make_vec_env(CircleEnv, n_envs=1, env_kwargs=dict())

# env = CircleEnv(render_mode="human", log_level="info", vehicles_to_spawn=5)
env = CircleEnv(render_mode=None, log_level="info", vehicles_to_spawn=5)

In [ ]:
from stable_baselines3.common.callbacks import BaseCallback

model_save_name = "A2C_model_dump_v0.3.0.zip"
log_folder = "./a2c_circle_tensorboard/"
log_name = "v0.3.0_demo5Vehicles"

class StepLoggerCallback(BaseCallback):
    def __init__(self, verbose=0):
        super(StepLoggerCallback, self).__init__(verbose)
    
    def _on_step(self) -> bool:
        self.logger.record("current_step", self.num_timesteps)
        self.logger.dump(self.num_timesteps)
        return True


# Train the agent
model = A2C("MultiInputPolicy", env, verbose=1, tensorboard_log=log_folder)
learning_steps = 10000
model.learn(learning_steps, callback=StepLoggerCallback(), tb_log_name=log_name)
model.save(model_save_name)

In [ ]:
# Quick evaluation
from stable_baselines3.common.evaluation import evaluate_policy
print("Training finished. Starting evaluation")
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=1)
print(mean_reward)
print(std_reward)

In [ ]:
# Cleanup
env.close()

In [ ]:
from stable_baselines3 import PPO, A2C, DQN
from gymtest import CircleEnv

# Observe execution of trained agent in GUI
env = CircleEnv(render_mode="human", log_level="info", vehicles_to_spawn=5)
model = A2C.load(model_save_name, env)

num_steps = 5
observation, info = env.reset()
for t in range(num_steps):
        actions, _ = model.predict(observation, state=None, deterministic=False)
        observation, reward, terminated, truncated, info = env.step(actions, log_level="info")

env.close()

In [ ]:
# Random actions to compare with the agent
env = CircleEnv(render_mode="human", log_level="info", vehicles_to_spawn=5)
observation, info = env.reset()
for _ in range(5):
    action = env.action_space.sample() # select a random action
    observation, reward, terminated, truncated, info = env.step(action)
    # if terminated or truncated:
        # observation, info = env.reset()
        
env.close()

---
# Test charging stop removal

In [ ]:
from circletest import Simulation

cs_id = "cs_0"
vehicle_id = "myVehicle0"
simulation = Simulation(gui=False)
simulation.add_vehicles()

def print_stops():
    stops = simulation.get_stops(vehicle_id)
    print(f"Stops: {stops}")

# starten
simulation.step()
print_stops()
# rerouten
print("REROUTE")
simulation.reroute_for_charging(vehicle_id, cs_id)
print_stops()
# stop removen
print("REMOVE STOP")
simulation.remove_charging_stop(vehicle_id)
print_stops()

simulation.close()